<a href="https://colab.research.google.com/github/MuskaanPrabhakar/Resume-Skill-Matcher/blob/main/resume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tools

In [ ]:
!pip install langchain_community

In [ ]:
!pip install pypdf

In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
)

In [ ]:
path='/content/drive'

In [ ]:
from google.colab import drive
drive.mount(path)
%cd enter exact folder path #move to a specifc directory

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
!pip install langchain_chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
from pathlib import Path

In [ ]:
from google.colab import files

In [ ]:
import os

#Loading and making chunks [documents] from folder
the document could be both text and pdf

In [ ]:
pdf_loader = DirectoryLoader(
    "enter exact folder path",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

In [ ]:
txt_loader = DirectoryLoader(
    "enter exact folder path",
    glob="**/*.txt",
    loader_cls=TextLoader,
)

In [ ]:
docs = pdf_loader.load() + txt_loader.load()

In [ ]:
chunks = splitter.split_documents(docs)

#Chunks into vector embeddings and into chroma

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

#To upload new resume to vector database

In [ ]:
# Upload file
uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]
save_path = f"enter exact file path{filename}"
with open(save_path, "wb") as f:
  f.write(uploaded[filename])

In [ ]:
if Path(save_path).suffix.lower() == ".pdf":
    loader = PyPDFLoader(save_path)
elif Path(save_path).suffix.lower() == ".txt":
    loader = TextLoader(save_path, encoding="utf-8")
new_docs = loader.load()
#docs+=new_docs
new_chunks = splitter.split_documents(new_docs)
#chunks+=new_chunks
vectorstore.add_documents(new_chunks)

In [ ]:
count = len(list(Path("enter exact file path").glob("*")))
print(count)
#to just verify

#Delete a file & vector embeddings


In [ ]:
filepath=(input("Enter a filepath: "))

In [ ]:
vectorstore._collection.delete(
        where={"source": filepath}
    )

In [ ]:
os.remove(filepath)

In [ ]:
count = len(list(Path("enter exact file path").glob("*")))
print(count)
#to just verify

#Query section

In [ ]:
query= input("What kind of skills you want for your job?: ")

In [ ]:
results = vectorstore.similarity_search_with_score(
    query,
    k=5
)

#Ouput

In [ ]:
print("Top 5 matched resumes from your folder: ")
for doc, score in results:
    print(f"File path: {doc.metadata["source"]} score:{score}")